In [2]:
import os
import json
import numpy as np
import pandas as pd
import sympy as sp

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
MODELSDIR = CONFIGS['filepaths']['models']
SRMODELS  = CONFIGS['experiments']['sr']['optimizedeqs']

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)

regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in regdf.iterrows()}

ORDER  = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}

TPMEAN = STATS['tp_mean']
TPSTD  = STATS['tp_std']
ZMIN   = (0.0 - TPMEAN) / TPSTD

print(f'tp: mean={TPMEAN:.6f}, std={TPSTD:.6f}, zmin={ZMIN:.4f}')
for var in ['rh','thetae','thetaestar','bl','shf','lhf']:
    m = STATS.get(f'{var}_mean')
    s = STATS.get(f'{var}_std')
    if m is not None:
        print(f'{var}: mean={m:.6f}, std={s:.6f}')

tp: mean=0.337616, std=0.528410, zmin=-0.6389
rh: mean=68.068031, std=21.958771
thetae: mean=342.388641, std=9.085335
thetaestar: mean=355.504700, std=12.706666
bl: mean=-0.075989, std=0.068279
shf: mean=13.909072, std=42.258709
lhf: mean=124.393448, std=70.937378


In [4]:
c = REGISTRY['sr_bl_eq']['constants']
blmean = STATS['bl_mean']
blstd  = STATS['bl_std']

threshold = blmean - c['a'] * blstd

print(f'SR-BL: raw = (bl_norm + {c["a"]:.4f})^3 + {c["b"]:.4f}')
print(f'Physical: raw = ((B_L - {threshold:.4f}) / {blstd:.6f})^3 + {c["b"]:.4f}')
print(f'Onset threshold: B_L = {threshold:.4f}')

SR-BL: raw = (bl_norm + 0.3000)^3 + 0.1400
Physical: raw = ((B_L - -0.0965) / 0.068279)^3 + 0.1400
Onset threshold: B_L = -0.0965


In [5]:
c = REGISTRY['sr_atm_eq']['constants']
rhmean  = STATS['rh_mean']
rhstd   = STATS['rh_std']
temean  = STATS['thetae_mean']
testd   = STATS['thetae_std']
tesmean = STATS['thetaestar_mean']
tesstd  = STATS['thetaestar_std']

tecoef  = c['b'] / testd
tescoef = c['b'] * 1.0 / tesstd
offset  = c['c'] + c['b'] * temean / testd - c['b'] * tesmean / tesstd

print(f'SR-ATM: raw = {REGISTRY["sr_atm_eq"]["form"]}')
print(f'Constants: {c}')
print()
print(f'Moisture pathway:    (RH - {rhmean:.4f}) / {rhstd:.4f}')
print(f'Instability pathway: {tecoef:.6f}*thetae - {tescoef:.6f}*thetaestar - {offset:.4f}')
print(f'thetae/thetaestar sensitivity ratio: {tecoef/tescoef:.4f}')

SR-ATM: raw = a*cube(max(rh,thetae-b*thetaestar-c))
Constants: {'a': 1.57, 'b': 1.48, 'c': 0.37}

Moisture pathway:    (RH - 68.0680) / 21.9588
Instability pathway: 0.162900*thetae - 0.116474*thetaestar - 14.7379
thetae/thetaestar sensitivity ratio: 1.3986


In [6]:
print(f'Prediction pipeline:')
print(f'  z = {ZMIN:.4f} + max(raw, 0)')
print(f'  P = exp(z * {TPSTD:.6f} + {TPMEAN:.6f}) - 1  [mm]')
print()
for name in ORDER:
    entry = REGISTRY[name]
    print(f'{LABELS[name]}: {entry["form"]}  {entry["constants"]}')

Prediction pipeline:
  z = -0.6389 + max(raw, 0)
  P = exp(z * 0.528410 + 0.337616) - 1  [mm]

SR-BL: cube(bl+a)+b  {'a': 0.3, 'b': 0.14}
SR-ATM: a*cube(max(rh,thetae-b*thetaestar-c))  {'a': 1.57, 'b': 1.48, 'c': 0.37}
SR-SFC: sr_atm_eq+a*shf*(b-lf)+c*lhf  {'a': 1.0, 'b': 0.74, 'c': 0.21}
SR-ALL: sr_atm_eq+(thetae+a*shf)*cube(b-lf)+c  {'a': 5.24, 'b': 0.73, 'c': -0.12}


In [7]:
c = REGISTRY['sr_atm_eq']['constants']
rhmean  = STATS['rh_mean']
rhstd   = STATS['rh_std']
temean  = STATS['thetae_mean']
testd   = STATS['thetae_std']
tesmean = STATS['thetaestar_mean']
tesstd  = STATS['thetaestar_std']

arh  = 1.0 / rhstd
ate  = c['b'] / testd
ates = c['b'] / tesstd
aconst = c['c'] + c['b'] * temean / testd - c['b'] * tesmean / tesstd - rhmean / rhstd

print(f'Regime boundary in normalized space:')
print(f'  rh_norm = thetae_norm - {c["b"]:.4f}*thetaestar_norm - {c["c"]:.4f}')
print()
print(f'Regime boundary in physical space:')
print(f'  {arh:.6f}*RH = {ate:.6f}*thetae - {ates:.6f}*thetaestar - {aconst:.4f}')

Regime boundary in normalized space:
  rh_norm = thetae_norm - 1.4800*thetaestar_norm - 0.3700

Regime boundary in physical space:
  0.045540*RH = 0.162900*thetae - 0.116474*thetaestar - 11.6381
